# Feature B — Review Sentiment

Goal: classify every review as positive, neutral, or negative, evaluate against manually checked **real reviews**, compare multiple approaches, tune the strongest candidates, select a final model, and summarize results by product and category.

Low-token design: star ratings provide weak training labels; Gemini is optional and used only on a small manual evaluation set.


## 1. Setup and feature-specific preprocessing


In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, f1_score, ConfusionMatrixDisplay
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

PROJECT = Path.cwd()
if not (PROJECT / 'analysis').exists(): PROJECT = PROJECT.parent
sys.path.insert(0, str(PROJECT))
from analysis.cleaning import load_clean
from analysis.labelling import create_manual_annotation_template, load_manual_annotations, summarize_sentiment
products, reviews = load_clean()
df = reviews.dropna(subset=['body_clean','star_sentiment']).copy()
print('Usable reviews:', len(df))
display(df[['review_id','asin','rating','body_clean','star_sentiment','language']])


**Why these actions:** empty text cannot be classified; stars are documented weak labels, not final truth; text and labels remain linked by stable `review_id`. Very short meaningful reviews are retained.


## 2. EDA and interpretation


In [ ]:
fig, axes = plt.subplots(1,3,figsize=(16,5))
df['star_sentiment'].value_counts().plot.bar(ax=axes[0], title='Weak-label imbalance')
df['word_count'].plot.hist(bins=50, ax=axes[1], title='Review length')
df['language'].value_counts().head(10).plot.bar(ax=axes[2], title='Languages')
plt.tight_layout(); plt.show()
display(df.groupby('star_sentiment')['word_count'].describe())
print('Interpretation: use macro-F1 and class balancing; accuracy alone would reward the majority positive class.')


## 3. Create genuine manual ground truth


In [ ]:
manual_path = PROJECT / 'data' / 'clean' / 'manual_sentiment_labels.csv'
if not manual_path.exists():
    create_manual_annotation_template(df, n_per_class=100, output_file=manual_path)
    print('STOP: manually fill human_sentiment in', manual_path)
else:
    print('Manual sheet exists:', manual_path)

# Run only after every human_sentiment cell is completed:
# manual = load_manual_annotations(manual_path)


Annotate the review text yourself as positive, neutral, or negative. Do not copy the star rating blindly. Split the completed sheet into a development half for model selection and an untouched final-test half.


## 4. Candidate development using weak labels


In [ ]:
X_train, X_holdout, y_train, y_holdout = train_test_split(
    df['body_clean'], df['star_sentiment'], test_size=0.2, random_state=42, stratify=df['star_sentiment']
)

vader = SentimentIntensityAnalyzer()
def vader_predict(texts):
    labels = []
    for text in texts:
        score = vader.polarity_scores(text)['compound']
        labels.append('positive' if score >= 0.05 else 'negative' if score <= -0.05 else 'neutral')
    return np.array(labels)

candidates = {
    'logistic_regression': Pipeline([
        ('tfidf', TfidfVectorizer(stop_words='english')),
        ('model', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42)),
    ]),
    'linear_svm': Pipeline([
        ('tfidf', TfidfVectorizer(stop_words='english')),
        ('model', LinearSVC(class_weight='balanced', random_state=42)),
    ]),
}

weak_results = [{'model':'vader', 'macro_f1': f1_score(y_holdout, vader_predict(X_holdout), average='macro')}]
for name, model in candidates.items():
    model.fit(X_train, y_train)
    weak_results.append({'model':name, 'macro_f1':f1_score(y_holdout, model.predict(X_holdout), average='macro')})
display(pd.DataFrame(weak_results).sort_values('macro_f1', ascending=False))


## 5. Fine-tune the strongest local candidates


In [ ]:
parameter_grid = {
    'tfidf__ngram_range': [(1,1),(1,2)],
    'tfidf__min_df': [1,2,3],
    'tfidf__sublinear_tf': [True,False],
    'model__C': [0.25,0.5,1,2],
}
tuned = {}
for name in ('logistic_regression','linear_svm'):
    search = GridSearchCV(candidates[name], parameter_grid, scoring='f1_macro', cv=3, n_jobs=-1)
    search.fit(X_train, y_train)
    tuned[name] = search.best_estimator_
    print(name, search.best_score_, search.best_params_)


## 6. Final selection using manually checked real reviews


In [ ]:
manual = load_manual_annotations(manual_path)
manual_dev, manual_test = train_test_split(
    manual, test_size=0.5, random_state=42, stratify=manual['human_sentiment']
)
review_lookup = df.set_index('review_id')['body_clean']

def score_on_manual(model, labels):
    text = labels['review_id'].map(review_lookup)
    pred = model.predict(text)
    return f1_score(labels['human_sentiment'], pred, average='macro'), pred

selection = []
for name, model in tuned.items():
    score, _ = score_on_manual(model, manual_dev)
    selection.append({'model':name, 'manual_dev_macro_f1':score})
selection.append({'model':'vader', 'manual_dev_macro_f1':f1_score(manual_dev['human_sentiment'], vader_predict(manual_dev['review_id'].map(review_lookup)), average='macro')})
selection = pd.DataFrame(selection).sort_values('manual_dev_macro_f1', ascending=False)
display(selection)
FINAL_NAME = selection.iloc[0]['model']
FINAL_MODEL = tuned.get(FINAL_NAME)
test_text = manual_test['review_id'].map(review_lookup)
test_pred = vader_predict(test_text) if FINAL_NAME == 'vader' else FINAL_MODEL.predict(test_text)
print('Untouched manual-test results for', FINAL_NAME)
print(classification_report(manual_test['human_sentiment'], test_pred, zero_division=0))
ConfusionMatrixDisplay.from_predictions(manual_test['human_sentiment'], test_pred); plt.show()


Optional Gemini comparison: use `GeminiLabeller(model_name='gemini-3.5-flash-lite', batch_size=25)` only on the manual test set. One 300-review sheet requires roughly 12 batched requests, rather than thousands of one-review calls.


## 7. Classify every review and create required summaries


In [ ]:
if FINAL_NAME == 'vader':
    df['predicted_sentiment'] = vader_predict(df['body_clean'])
else:
    # Retrain the selected configuration on all weak-labelled reviews.
    FINAL_MODEL.fit(df['body_clean'], df['star_sentiment'])
    df['predicted_sentiment'] = FINAL_MODEL.predict(df['body_clean'])

product_summary, category_summary = summarize_sentiment(df, 'predicted_sentiment')
results_dir = PROJECT / 'data' / 'models' / 'ReviewSentiment'
results_dir.mkdir(parents=True, exist_ok=True)
df.to_parquet(results_dir / 'labelled_reviews.parquet', index=False)
product_summary.to_csv(results_dir / 'product_sentiment_summary.csv', index=False)
category_summary.to_csv(results_dir / 'category_sentiment_summary.csv', index=False)
display(product_summary.head())
display(category_summary.head())


## 8. Final interpretation

Record the selected model, tuned parameters, manual-development macro-F1, untouched manual-test macro-F1, class-level weaknesses, and any category/language patterns before submission.
